In [ ]:
"""Responsibilities: register a frozen experiment module.
Implementation: compile source in an isolated module namespace.
Related Modules: later notebook cells import this registered module.
Declaration Index: None.
Variable Index: source is embedded Python; module is the registered namespace.
"""
import sys
import types
source = '"""Responsibilities: run an independent fixed-protocol feasibility study for a dual-head recommendation\nmodel on a Kaggle GPU.\nImplementation: download a fixed data release, consolidate consistent duplicate pairs, split by job,\ncompare text and structured baselines, train a dual-head network on frozen Qwen embeddings, and assess\ntop-10 qualification reranking.\nRelated Modules: this historical source does not import or alter AiInterviewer business modules;\nKaggle kernel metadata controls private remote execution.\n\nDeclaration Index:\n- write_json: atomically write audit records and results.\n- bootstrap: install the specified Transformers version, check CUDA, and record the environment.\n- download_data: download fixed-version CSVs and record SHA256 hashes.\n- consolidate_labels: validate and merge duplicate labels while preserving each task\'s missing state.\n- split_jobs: create a fixed, mutually exclusive job split.\n- parse_list: parse Python-style list values from the data.\n- profile_texts: construct model inputs without identity or labels.\n- structured_features: compute pairwise features with explicit semantics.\n- prepare_data: verify entities, labels, distributions, and split separation.\n- ranking_metrics: compute ranking and query-coverage metrics on the explicitly labeled candidate pool.\n- evaluate_scores: compute AP, AUC, and ranking metrics for both tasks.\n- lexical_baseline: fit TF-IDF on training-side text only.\n- encode_profiles: encode with a fixed token limit and record truncation rates.\n- build_pair_features: combine frozen embeddings with structured inputs.\n- DualHeadModel: define shared representation and separate preference/qualification heads.\n- DualHeadModel.__init__: create the fixed network layers.\n- DualHeadModel.forward: produce two uncalibrated logits.\n- train_dual_head: train with label masks and select checkpoints by validation AP only.\n- structured_baseline: fit training-side standardization and two logistic regressions.\n- rerank_qualification: rerank the semantic-retrieval top 10 for each test job.\n- write_report: save readable metrics, boundaries, and configuration.\n- main: run stages in dependency order and record/re-raise failures.\n\nVariable Index:\n- CONFIG: fixed data release, model revision, split, preprocessing, and training configuration.\n- FILES: seven public synthetic data files allowed for download.\n- STRUCTURED_NAMES: ordered structured feature names and meanings.\n- LOGGER: records stage, size, statistics, and errors without credentials or profile text.\n\nConstraints: CUDA is required; do not switch models, shrink inputs, or retry failed operations. Results\nare from a synthetic-data new-job experiment; they do not reproduce the authors\' original split, and\nunobserved labels are not treated as negative."""\n\nfrom __future__ import annotations\n\nimport ast\nimport copy\nimport gc\nimport hashlib\nimport importlib.metadata\nimport json\nimport logging\nimport os\nimport platform\nimport random\nimport subprocess\nimport sys\nimport time\nimport urllib.request\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import average_precision_score, roc_auc_score\nfrom sklearn.preprocessing import StandardScaler\n\nCONFIG = {\n    "protocol": "jobrec-cold-job-feasibility-v1",\n    "seed": 42,\n    "data_commit": "c742feea3730e8d34ec2e8ae7e58c8c0ee8a53fd",\n    "embedding_model": "Qwen/Qwen3-Embedding-0.6B",\n    "embedding_revision": "97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3",\n    "reranker_model": "Qwen/Qwen3-Reranker-0.6B",\n    "reranker_revision": "e61197ed45024b0ed8a2d74b80b4d909f1255473",\n    "transformers_version": "4.56.2",\n    "job_split_counts": [70, 15, 15],\n    "embedding_max_length": 384,\n    "embedding_batch_size": 8,\n    "embedding_dtype": "float16",\n    "attention": "sdpa",\n    "job_instruction": "Given a job description, retrieve candidates qualified for the job.",\n    "epochs": 12,\n    "training_batch_size": 512,\n    "learning_rate": 0.001,\n    "weight_decay": 0.0001,\n    "hidden_sizes": [128, 32],\n    "dropout": 0.1,\n    "tfidf_max_features": 20000,\n    "tfidf_min_df": 2,\n    "logistic_max_iter": 2000,\n    "rerank_top_k": 10,\n    "rerank_batch_size": 4,\n    "rerank_query_tokens": 160,\n    "rerank_document_tokens": 320,\n    "rerank_max_length": 768,\n    "ranking_k": [5, 10],\n    "selection_metric": "mean_validation_average_precision_over_two_observed_tasks",\n}\nFILES = (\n    "Final_users.csv", "Final_items.csv", "train.csv", "val.csv", "test.csv",\n    "qual_train.csv", "qual_test.csv",\n)\nSTRUCTURED_NAMES = (\n    "skill_coverage", "interest_matches_industry", "major_accepted", "work_mode_matches",\n    "gpa_margin", "experience_months_margin", "academic_level_margin",\n    "weekly_hours_margin", "commitment_months_margin", "publications_margin",\n    "summer_commitment",\n)\nLOGGER = logging.getLogger("jobrec")\n\n\ndef write_json(path: Path, value: object) -> None:\n    """Functionality: atomically save a JSON value.\n    Inputs: destination path and JSON-serializable value. Outputs: none; writes the file.\n    Logic: write a temporary sibling file, then replace the destination.\n    Constraints: reject nonfinite values; propagate write failures."""\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")\n    temporary.replace(path)\n\n\ndef bootstrap(output: Path) -> None:\n    """Functionality: install the fixed dependencies and verify the GPU environment.\n    Inputs: output directory. Outputs: none.\n    Logic: install the pinned Transformers version, set seeds, and record platform package versions.\n    Constraints: fail when CUDA is unavailable; do not change hardware or parameters. Retain and record\n    Kaggle-preinstalled Torch/NumPy; this is not a fully locked environment."""\n    os.environ["HF_HOME"] = "/tmp/jobrec_hf"\n    os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"\n    os.environ["HF_HUB_DISABLE_XET"] = "1"\n    os.environ["TOKENIZERS_PARALLELISM"] = "false"\n    subprocess.run([\n        sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check",\n        "transformers==" + CONFIG["transformers_version"],\n    ], check=True)\n    if not torch.cuda.is_available():\n        raise RuntimeError("Protocol requires CUDA; refusing a CPU substitution.")\n    random.seed(CONFIG["seed"])\n    np.random.seed(CONFIG["seed"])\n    torch.manual_seed(CONFIG["seed"])\n    torch.cuda.manual_seed_all(CONFIG["seed"])\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n    torch.set_num_threads(2)\n    environment = {\n        "python": sys.version, "platform": platform.platform(),\n        "packages": {name: importlib.metadata.version(name) for name in (\n            "torch", "numpy", "pandas", "scikit-learn", "transformers", "tokenizers",\n            "huggingface-hub", "safetensors",\n        )},\n        "gpu_names": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],\n        "used_device": "cuda:0", "cuda": torch.version.cuda,\n        "note": "Seeds fixed; bitwise reproducibility across GPU/library versions is not guaranteed.",\n    }\n    write_json(output / "environment.json", environment)\n    LOGGER.info("Environment ready: GPUs=%s", environment["gpu_names"])\n\n\ndef download_data(output: Path) -> dict[str, pd.DataFrame]:\n    """Functionality: obtain the seven CSVs from the fixed data release.\n    Inputs: output directory. Outputs: mapping from file names to DataFrames.\n    Logic: request public URLs and record each file\'s hash and row count.\n    Constraints: no retries; do not write credentials or a duplicate of raw profile text."""\n    import io\n\n    frames, manifest = {}, {}\n    base = "https://raw.githubusercontent.com/brycekan123/DualOptimization_jobrec/"\n    for filename in FILES:\n        url = base + CONFIG["data_commit"] + "/dataset/" + filename\n        with urllib.request.urlopen(url, timeout=60) as response:\n            raw = response.read()\n        frame = pd.read_csv(io.BytesIO(raw))\n        frames[filename] = frame\n        manifest[filename] = {"url": url, "sha256": hashlib.sha256(raw).hexdigest(), "rows": len(frame)}\n        LOGGER.info("Downloaded %s: %d rows", filename, len(frame))\n    write_json(output / "data_manifest.json", manifest)\n    return frames\n\n\ndef consolidate_labels(frames: dict[str, pd.DataFrame]) -> tuple[pd.DataFrame, dict]:\n    """Functionality: construct the unique pair table and audit raw duplicates.\n    Inputs: source CSV DataFrames. Outputs: merged pair table and duplicate audit.\n    Logic: retain only consistent labels for the same task/pair; outer-join tasks to preserve NaN masks.\n    Constraints: conflicting or nonbinary labels raise an error; ignore source scores; never fill missing\n    qualification labels with zero."""\n    tables, audit = [], {}\n    for task, names in (("pref", ("train.csv", "val.csv", "test.csv")),\n                        ("qual", ("qual_train.csv", "qual_test.csv"))):\n        label = "label_" + task\n        combined = pd.concat([frames[name][["user", "item", label]] for name in names], ignore_index=True)\n        if combined.isna().any().any() or not set(combined[label].unique()).issubset({0, 1}):\n            raise ValueError(f"Invalid identifiers or binary labels for {task}")\n        if (combined.groupby(["user", "item"])[label].nunique() > 1).any():\n            raise ValueError(f"Conflicting duplicate labels for {task}")\n        unique = combined.drop_duplicates(["user", "item"]).copy()\n        audit[task] = {"source_rows": len(combined), "unique_pairs": len(unique),\n                       "consistent_duplicates_collapsed": len(combined) - len(unique)}\n        tables.append(unique)\n    merged = tables[0].merge(tables[1], on=["user", "item"], how="outer", validate="one_to_one")\n    return merged.sort_values(["user", "item"]).reset_index(drop=True), audit\n\n\ndef split_jobs(job_ids: list[str]) -> dict[str, str]:\n    """Functionality: assign a fixed cold-job split.\n    Inputs: job IDs. Outputs: mapping from job IDs to split names.\n    Logic: sort IDs, then apply an independent seeded permutation.\n    Constraints: require exactly 100 distinct jobs and fixed 70/15/15 proportions."""\n    ids = sorted(job_ids)\n    if len(ids) != 100 or len(set(ids)) != 100:\n        raise ValueError("Expected exactly 100 distinct jobs.")\n    shuffled = np.random.default_rng(CONFIG["seed"]).permutation(ids)\n    return {str(job): ("train" if i < 70 else "validation" if i < 85 else "test")\n            for i, job in enumerate(shuffled)}\n\n\ndef parse_list(value: str) -> list[str]:\n    """Functionality: parse one source-data list value.\n    Inputs: Python list text. Outputs: list of strings.\n    Logic: use literal_eval and validate the parsed type.\n    Constraints: malformed values raise an error; do not guess alternate separators."""\n    parsed = ast.literal_eval(value)\n    if not isinstance(parsed, list) or not all(isinstance(item, str) for item in parsed):\n        raise ValueError("Expected a list of strings in profile.")\n    return parsed\n\n\ndef profile_texts(users: pd.DataFrame, jobs: pd.DataFrame) -> tuple[list[str], list[str]]:\n    """Functionality: construct model text for candidate profiles and job queries.\n    Inputs: complete profile tables. Outputs: candidate texts and job-query texts.\n    Logic: place skills/interests/education/experience first and synthetic descriptions afterward; use a\n    fixed qualification-retrieval instruction for jobs.\n    Constraints: exclude IDs, match labels, source scores, and fictional interview evidence; later truncate\n    to the fixed token limit."""\n    candidate_texts = [\n        f"Skills: {\', \'.join(parse_list(row.skills))}. Interests: {\', \'.join(parse_list(row.interests))}. "\n        f"Education: {row.academic_level}, {row.masters_major}, {row.undergrad_major}. "\n        f"Experience months: {row.months_experience}. Profile: {row.student_llm_output}"\n        for row in users.itertuples()\n    ]\n    job_texts = [\n        f"Instruct: {CONFIG[\'job_instruction\']}\\nQuery: "\n        f"Industry: {row.industry}. Skills: {\', \'.join(parse_list(row.required_skills))}. "\n        f"Education: {row.min_academic_level}. Experience months: {row.min_months_experience}. "\n        f"Description: {row.job_llm_output}"\n        for row in jobs.itertuples()\n    ]\n    return candidate_texts, job_texts\n\n\ndef structured_features(pairs: pd.DataFrame, users: pd.DataFrame, jobs: pd.DataFrame) -> np.ndarray:\n    """Functionality: calculate eleven explicit match features.\n    Inputs: pair table and profile tables. Outputs: float32 feature matrix.\n    Logic: compute skill coverage, preference/major/work-mode matches, and numeric margins.\n    Constraints: qualification labels may be generated from similar rules, so this baseline measures\n    learnability on synthetic tasks only. It contains no demographic attributes or nonexistent interview\n    scores."""\n    user_map = users.set_index("student_id").to_dict("index")\n    job_map = jobs.set_index("job_id").to_dict("index")\n    levels = {"UG1": 1, "UG2": 2, "UG3": 3, "UG4": 4, "MS1": 5, "MS2": 6,\n              "PhD1": 7, "PhD2": 8, "PhD3": 9, "PhD4": 10, "PhD5": 11}\n    result = []\n    for row in pairs.itertuples():\n        user, job = user_map[row.user], job_map[row.item]\n        skills, required = set(parse_list(user["skills"])), set(parse_list(job["required_skills"]))\n        if not required:\n            raise ValueError("Empty required skill list is outside the frozen protocol.")\n        majors = {user["masters_major"], user["second_major"], user["undergrad_major"]}\n        features = [len(skills & required) / len(required),\n                    float(job["industry"] in parse_list(user["interests"])),\n                    float(bool(majors & set(parse_list(job["acceptable_majors"])))),\n                    float(user["in_person_commitment"] == job["job_in_person_commitment"]),\n                    float(user["gpa"]) - float(job["min_gpa"]),\n                    float(user["months_experience"]) - float(job["min_months_experience"]),\n                    float(levels[user["academic_level"]] - levels[job["min_academic_level"]]),\n                    float(user["hours_per_week"]) - float(job["min_hours_per_week"]),\n                    float(user["length_of_commitment"]) - float(job["min_length_of_commitment"]),\n                    float(user["num_publications"]) - float(job["min_num_publications"]),\n                    float(str(user["commit_to_summer"]).lower() == "true")]\n        result.append(features)\n    matrix = np.asarray(result, dtype=np.float32)\n    if not np.isfinite(matrix).all():\n        raise ValueError("Non-finite structured features.")\n    return matrix\n\n\ndef prepare_data(frames: dict[str, pd.DataFrame], output: Path) -> tuple:\n    """Functionality: map entities and audit the split.\n    Inputs: source tables and output path. Outputs: all prepared training objects.\n    Logic: merge pairs, split by job, and verify that each task/split has positive and negative labels.\n    Constraints: candidates may cross splits, so this is a cold-job—not cold-candidate—study. Candidate\n    pools contain only source pairs with explicit labels."""\n    users = frames["Final_users.csv"].sort_values("student_id").reset_index(drop=True)\n    jobs = frames["Final_items.csv"].sort_values("job_id").reset_index(drop=True)\n    if users.student_id.duplicated().any() or jobs.job_id.duplicated().any():\n        raise ValueError("Entity IDs must be unique.")\n    pairs, audit = consolidate_labels(frames)\n    split_map = split_jobs(jobs.job_id.tolist())\n    pairs["split"] = pairs.item.map(split_map)\n    pairs["u_idx"] = pairs.user.map(dict(zip(users.student_id, users.index)))\n    pairs["j_idx"] = pairs.item.map(dict(zip(jobs.job_id, jobs.index)))\n    if pairs[["split", "u_idx", "j_idx"]].isna().any().any():\n        raise ValueError("Unknown entity in label table.")\n    for split in ("train", "validation", "test"):\n        audit[split] = {}\n        for task in ("pref", "qual"):\n            observed = pairs.loc[pairs.split.eq(split), "label_" + task].dropna()\n            if set(observed.unique()) != {0, 1}:\n                raise ValueError(f"Both classes required: {split}/{task}")\n            audit[split][task] = {"pairs": len(observed), "positives": int(observed.sum())}\n    audit["protocol_note"] = "Disjoint jobs; shared candidates; observed labeled candidate pools only."\n    audit["structured_feature_names"] = STRUCTURED_NAMES\n    write_json(output / "split_manifest.json", split_map)\n    write_json(output / "data_audit.json", audit)\n    pairs.to_csv(output / "pair_splits.csv", index=False)\n    texts = profile_texts(users, jobs)\n    features = structured_features(pairs, users, jobs)\n    LOGGER.info("Data audit: %s", json.dumps(audit))\n    return pairs, users, jobs, texts, features\n\n\ndef ranking_metrics(frame: pd.DataFrame, scores: np.ndarray, task: str) -> dict:\n    """Functionality: compute macro-averaged ranking metrics on explicitly labeled candidate pools.\n    Inputs: observed-label rows, scores, and task. Outputs: metrics dictionary.\n    Logic: group preference rows by candidate and qualification rows by job; exclude groups without\n    positives from the mean and report them separately; break ties by stable input order.\n    Constraints: this is not full-corpus retrieval; nDCG/Recall denominators use only labeled candidates."""\n    scored = frame.assign(pred=np.asarray(scores))\n    group_column = "user" if task == "pref" else "item"\n    label_column = "label_" + task\n    per_query, pool_sizes, all_negative = [], [], 0\n    for _, group in scored.groupby(group_column, sort=True):\n        labels = group[label_column].to_numpy()\n        positives = int(labels.sum())\n        if positives == 0:\n            all_negative += 1\n            continue\n        ranked = group.sort_values("pred", ascending=False, kind="stable")[label_column].to_numpy()\n        values = {}\n        for k in CONFIG["ranking_k"]:\n            limit = min(k, len(ranked))\n            discount = 1 / np.log2(np.arange(limit) + 2)\n            ideal = (1 / np.log2(np.arange(min(positives, limit)) + 2)).sum()\n            values[f"ndcg@{k}"] = float((ranked[:limit] * discount).sum() / ideal)\n            values[f"recall@{k}"] = float(ranked[:limit].sum() / positives)\n        per_query.append(values)\n        pool_sizes.append(len(group))\n    if not per_query:\n        raise ValueError(f"No evaluable positive queries for {task}")\n    return {**{name: float(np.mean([row[name] for row in per_query])) for name in per_query[0]},\n            "positive_queries": len(per_query), "all_negative_queries_excluded": all_negative,\n            "median_labeled_pool": float(np.median(pool_sizes))}\n\n\ndef evaluate_scores(pairs: pd.DataFrame, scores: np.ndarray, split: str) -> dict:\n    """Functionality: evaluate both tasks on one data split.\n    Inputs: pair table, N-by-2 scores, and split name. Outputs: per-task metrics.\n    Logic: use explicit labels only.\n    Constraints: sigmoid/ranking scores are not calibrated business probabilities."""\n    result = {}\n    for column, task in enumerate(("pref", "qual")):\n        selected = pairs.split.eq(split) & pairs["label_" + task].notna()\n        frame = pairs.loc[selected]\n        labels = frame["label_" + task].to_numpy()\n        values = scores[selected.to_numpy(), column]\n        result[task] = {"average_precision": float(average_precision_score(labels, values)),\n                        "roc_auc": float(roc_auc_score(labels, values)),\n                        "positive_prevalence": float(labels.mean()), "pairs": len(frame),\n                        **ranking_metrics(frame, values, task)}\n    return result\n\n\ndef lexical_baseline(pairs: pd.DataFrame, texts: tuple, output: Path) -> np.ndarray:\n    """Functionality: calculate TF-IDF pairwise cosine scores.\n    Inputs: pairs, text representations, and output directory. Outputs: shared semantic scores for N pairs.\n    Logic: fit vocabulary only on entities appearing in training pairs; transform validation/test only.\n    Constraints: these scores do not learn bilateral preference; record vocabulary size without serializing\n    the profile vocabulary."""\n    candidate_texts, job_texts = texts\n    train = pairs.loc[pairs.split.eq("train")]\n    corpus = [candidate_texts[i] for i in sorted(train.u_idx.unique())]\n    corpus += [job_texts[i] for i in sorted(train.j_idx.unique())]\n    vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=CONFIG["tfidf_min_df"],\n                                 max_features=CONFIG["tfidf_max_features"], dtype=np.float32)\n    vectorizer.fit(corpus)\n    u, j = vectorizer.transform(candidate_texts), vectorizer.transform(job_texts)\n    values = np.asarray(u[pairs.u_idx].multiply(j[pairs.j_idx]).sum(axis=1)).ravel()\n    write_json(output / "lexical_audit.json", {"vocabulary_size": len(vectorizer.vocabulary_),\n               "training_entities_only": True})\n    return np.column_stack([values, values])\n\n\ndef encode_profiles(texts: tuple, output: Path) -> tuple[np.ndarray, np.ndarray]:\n    """Functionality: produce frozen Qwen representations.\n    Inputs: candidate/job text and output directory. Outputs: normalized embeddings for both sides.\n    Logic: use the fixed revision, left padding, last-valid-token pooling, FP16, SDPA, and cuda:0.\n    Constraints: retain the 384-token limit and fail rather than changing it after memory errors; save only\n    embeddings and truncation statistics, never profile text."""\n    from transformers import AutoModel, AutoTokenizer\n\n    tokenizer = AutoTokenizer.from_pretrained(CONFIG["embedding_model"],\n                 revision=CONFIG["embedding_revision"], padding_side="left")\n    model = AutoModel.from_pretrained(CONFIG["embedding_model"],\n             revision=CONFIG["embedding_revision"], torch_dtype=torch.float16,\n             attn_implementation=CONFIG["attention"]).to("cuda:0").eval()\n    all_texts = texts[0] + texts[1]\n    chunks, lengths, started = [], [], time.monotonic()\n    batch_size = CONFIG["embedding_batch_size"]\n    for offset in range(0, len(all_texts), batch_size):\n        batch = all_texts[offset:offset + batch_size]\n        raw_tokens = tokenizer(batch, truncation=False, add_special_tokens=True)["input_ids"]\n        lengths.extend(len(row) for row in raw_tokens)\n        inputs = tokenizer(batch, padding=True, truncation=True,\n                  max_length=CONFIG["embedding_max_length"], return_tensors="pt").to("cuda:0")\n        with torch.inference_mode():\n            hidden = model(**inputs).last_hidden_state[:, -1]\n            encoded = torch.nn.functional.normalize(hidden.float(), dim=-1)\n        chunks.append(encoded.cpu().numpy())\n        if offset % (batch_size * 25) == 0:\n            LOGGER.info("Embedding %d/%d elapsed=%.1fs", offset + len(batch), len(all_texts), time.monotonic() - started)\n    embeddings = np.concatenate(chunks)\n    if not np.isfinite(embeddings).all():\n        raise ValueError("Non-finite embeddings")\n    audit = {"entities": len(all_texts), "dimension": embeddings.shape[1],\n             "elapsed_seconds": time.monotonic() - started,\n             "truncated_entities": int(np.sum(np.asarray(lengths) > CONFIG["embedding_max_length"])),\n             "median_original_tokens": float(np.median(lengths)),\n             "semantic_baseline": "job-query instruction; same qualification-oriented cosine in both directions"}\n    write_json(output / "embedding_audit.json", audit)\n    np.savez_compressed(output / "embeddings.npz", candidates=embeddings[:len(texts[0])], jobs=embeddings[len(texts[0]):])\n    del model, inputs, hidden, encoded\n    gc.collect()\n    torch.cuda.empty_cache()\n    LOGGER.info("Embedding completed: %s", audit)\n    return embeddings[:len(texts[0])], embeddings[len(texts[0]):]\n\n\ndef build_pair_features(pairs: pd.DataFrame, embeddings: tuple, structured: np.ndarray,\n                        output: Path) -> tuple[np.ndarray, np.ndarray]:\n    """Functionality: combine frozen embeddings and structured pair features.\n    Inputs: pair table, embeddings, features, and output directory. Outputs: network inputs and cosine scores.\n    Logic: concatenate absolute embedding differences, elementwise products, and training-side\n    standardized structured features.\n    Constraints: fit standardization on training rows only."""\n    u, j = embeddings[0][pairs.u_idx], embeddings[1][pairs.j_idx]\n    scaler = StandardScaler().fit(structured[pairs.split.eq("train")])\n    features = np.concatenate([np.abs(u - j), u * j, scaler.transform(structured)], axis=1).astype(np.float32)\n    cosine = (u * j).sum(axis=1)\n    write_json(output / "feature_scaler.json", {"names": STRUCTURED_NAMES,\n               "mean": scaler.mean_.tolist(), "scale": scaler.scale_.tolist()})\n    return features, np.column_stack([cosine, cosine])\n\n\nclass DualHeadModel(torch.nn.Module):\n    """Functionality: define the shared-representation preference/qualification network.\n    Logic: use an MLP trunk followed by one linear output per task.\n    Constraints: the heads use independent observed-label masks; instance state contains trained weights,\n    not entity IDs."""\n\n    def __init__(self, feature_size: int):\n        """Functionality: initialize the network.\n        Inputs: feature dimension. Outputs: a new model instance.\n        Logic: create fixed 128- and 32-unit hidden layers with dropout.\n        Constraints: layer parameters must match CONFIG."""\n        super().__init__()\n        first, second = CONFIG["hidden_sizes"]\n        self.trunk = torch.nn.Sequential(torch.nn.Linear(feature_size, first), torch.nn.ReLU(),\n                     torch.nn.Dropout(CONFIG["dropout"]), torch.nn.Linear(first, second), torch.nn.ReLU())\n        self.heads = torch.nn.ModuleList([torch.nn.Linear(second, 1), torch.nn.Linear(second, 1)])\n\n    def forward(self, features: torch.Tensor) -> torch.Tensor:\n        """Functionality: produce preference and qualification logits.\n        Inputs: B-by-D feature tensor. Outputs: B-by-2 logits.\n        Logic: apply the shared trunk and concatenate the two linear heads.\n        Constraints: the inference caller is responsible for eval mode and sigmoid conversion."""\n        hidden = self.trunk(features)\n        return torch.cat([head(hidden) for head in self.heads], dim=1)\n\n\ndef train_dual_head(pairs: pd.DataFrame, features: np.ndarray, output: Path) -> np.ndarray:\n    """Functionality: train the dual-head model and score every pair.\n    Inputs: pair table, features, and output directory. Outputs: N-by-2 scores.\n    Logic: mask missing labels, use training-side positive-class weights, and select the best of 12 epochs\n    by mean validation AP across both tasks.\n    Constraints: never select from test results; no early stopping or automatic parameter changes; compute\n    test metrics only after training is complete."""\n    model = DualHeadModel(features.shape[1]).to("cuda:0")\n    x = torch.from_numpy(features).to("cuda:0")\n    targets = pairs[["label_pref", "label_qual"]].to_numpy(dtype=np.float32)\n    observed = np.isfinite(targets)\n    y = torch.from_numpy(np.nan_to_num(targets, nan=0)).to("cuda:0")\n    mask = torch.from_numpy(observed).to("cuda:0")\n    train_ids = np.flatnonzero(pairs.split.eq("train"))\n    val_ids = np.flatnonzero(pairs.split.eq("validation"))\n    weights = []\n    for column in range(2):\n        labels = targets[train_ids, column]\n        labels = labels[np.isfinite(labels)]\n        weights.append(float((labels == 0).sum() / (labels == 1).sum()))\n    criterion = torch.nn.BCEWithLogitsLoss(reduction="none", pos_weight=torch.tensor(weights, device="cuda:0"))\n    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["learning_rate"], weight_decay=CONFIG["weight_decay"])\n    best_score, best_state, history = -1.0, None, []\n    rng = np.random.default_rng(CONFIG["seed"])\n    for epoch in range(1, CONFIG["epochs"] + 1):\n        model.train()\n        losses, order = [], rng.permutation(train_ids)\n        for start in range(0, len(order), CONFIG["training_batch_size"]):\n            ids = order[start:start + CONFIG["training_batch_size"]]\n            optimizer.zero_grad(set_to_none=True)\n            element_loss = criterion(model(x[ids]), y[ids])\n            task_losses = [(element_loss[:, c] * mask[ids, c]).sum() / mask[ids, c].sum()\n                           for c in range(2) if mask[ids, c].any()]\n            loss = torch.stack(task_losses).mean()\n            loss.backward()\n            optimizer.step()\n            losses.append(float(loss.item()))\n        model.eval()\n        with torch.inference_mode():\n            val_scores = torch.sigmoid(model(x[val_ids])).cpu().numpy()\n        aps = [float(average_precision_score(targets[val_ids, c][observed[val_ids, c]],\n               val_scores[:, c][observed[val_ids, c]])) for c in range(2)]\n        validation_mean = float(np.mean(aps))\n        row = {"epoch": epoch, "mean_batch_loss": float(np.mean(losses)), "validation_ap": aps}\n        history.append(row)\n        LOGGER.info("Training epoch: %s", row)\n        if validation_mean > best_score:\n            best_score = validation_mean\n            best_state = copy.deepcopy(model.state_dict())\n            selected_epoch = epoch\n    if best_state is None:\n        raise RuntimeError("No finite validation checkpoint")\n    model.load_state_dict(best_state)\n    torch.save({"state_dict": {k: v.cpu() for k, v in best_state.items()},\n                "feature_size": features.shape[1], "config": CONFIG}, output / "dual_head.pt")\n    with torch.inference_mode():\n        scores = torch.sigmoid(model(x)).cpu().numpy()\n    write_json(output / "training_history.json", {"epochs": history, "selected_epoch": selected_epoch,\n               "selection_validation_mean_ap": best_score, "positive_weights": weights})\n    del model, x, y, mask, optimizer, best_state\n    gc.collect()\n    torch.cuda.empty_cache()\n    return scores\n\n\ndef structured_baseline(pairs: pd.DataFrame, features: np.ndarray, output: Path) -> np.ndarray:\n    """Functionality: fit structured logistic-regression baselines.\n    Inputs: pairs, raw 11-dimensional features, and output directory. Outputs: N-by-2 scores.\n    Logic: fit scaling and logistic regression for each task using only that task\'s training labels.\n    Constraints: no label propagation between tasks."""\n    predictions, records = [], {}\n    for task in ("pref", "qual"):\n        observed = pairs.split.eq("train") & pairs["label_" + task].notna()\n        scaler = StandardScaler().fit(features[observed])\n        x = scaler.transform(features)\n        model = LogisticRegression(class_weight="balanced", random_state=CONFIG["seed"],\n                                    max_iter=CONFIG["logistic_max_iter"])\n        model.fit(x[observed], pairs.loc[observed, "label_" + task])\n        if int(model.n_iter_[0]) >= CONFIG["logistic_max_iter"]:\n            raise RuntimeError(f"Logistic regression did not converge: {task}")\n        predictions.append(model.predict_proba(x)[:, 1])\n        records[task] = {"names": STRUCTURED_NAMES, "coef": model.coef_[0].tolist(),\n                         "intercept": model.intercept_.tolist(), "iterations": model.n_iter_.tolist()}\n    write_json(output / "structured_logistic.json", records)\n    return np.column_stack(predictions)\n\n\ndef rerank_qualification(pairs: pd.DataFrame, texts: tuple, cosine: np.ndarray, output: Path) -> dict:\n    """Functionality: run two-stage reranking for qualification.\n    Inputs: pairs, text, cosine scores, and output directory. Outputs: metrics on the fixed test candidate pool.\n    Logic: retrieve the top 10 by semantic score per job, rerank the prefix with Qwen yes/no logits, and\n    preserve the original order for the tail.\n    Constraints: query/document limits are fixed at 160/320 tokens; this is not full-corpus retrieval; do\n    not report a pseudo-probability AUC or modify the candidate pool."""\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n\n    frame = pairs.loc[pairs.split.eq("test") & pairs.label_qual.notna()].copy()\n    frame["semantic"] = cosine[frame.index, 1]\n    tokenizer = AutoTokenizer.from_pretrained(CONFIG["reranker_model"],\n                 revision=CONFIG["reranker_revision"], padding_side="left")\n    model = AutoModelForCausalLM.from_pretrained(CONFIG["reranker_model"],\n             revision=CONFIG["reranker_revision"], torch_dtype=torch.float16,\n             attn_implementation=CONFIG["attention"]).to("cuda:0").eval()\n    prefix = (\'<|im_start|>system\\nJudge whether the Document meets the requirements based on the Query \'\n              \'and the Instruct provided. Note that the answer can only be "yes" or "no".<|im_end|>\\n<|im_start|>user\\n\')\n    suffix = "<|im_end|>\\n<|im_start|>assistant\\n<think>\\n\\n</think>\\n\\n"\n    prefix_ids, suffix_ids = tokenizer.encode(prefix, add_special_tokens=False), tokenizer.encode(suffix, add_special_tokens=False)\n    selected, payloads = [], []\n    ranked_groups = []\n    for _, group in frame.groupby("item", sort=True):\n        ranked = group.sort_values("semantic", ascending=False, kind="stable")\n        ranked_groups.append(ranked.index.tolist())\n        for row in ranked.head(CONFIG["rerank_top_k"]).itertuples():\n            query_ids = tokenizer.encode(texts[1][row.j_idx], add_special_tokens=False)[:CONFIG["rerank_query_tokens"]]\n            doc_ids = tokenizer.encode(texts[0][row.u_idx], add_special_tokens=False)[:CONFIG["rerank_document_tokens"]]\n            body = f"<Instruct>: {CONFIG[\'job_instruction\']}\\n<Query>: {tokenizer.decode(query_ids)}\\n<Document>: {tokenizer.decode(doc_ids)}"\n            ids = prefix_ids + tokenizer.encode(body, add_special_tokens=False) + suffix_ids\n            if len(ids) > CONFIG["rerank_max_length"]:\n                raise ValueError("Reranker formatted sequence exceeds frozen limit")\n            selected.append(row.Index)\n            payloads.append(ids)\n    values, started = [], time.monotonic()\n    yes, no = tokenizer.convert_tokens_to_ids("yes"), tokenizer.convert_tokens_to_ids("no")\n    for start in range(0, len(payloads), CONFIG["rerank_batch_size"]):\n        encoded = tokenizer.pad({"input_ids": payloads[start:start + CONFIG["rerank_batch_size"]]},\n                                padding=True, return_tensors="pt").to("cuda:0")\n        with torch.inference_mode():\n            logits = model(**encoded, logits_to_keep=1).logits[:, -1, :]\n            scores = torch.softmax(logits[:, [no, yes]].float(), dim=-1)[:, 1]\n        values.extend(scores.cpu().tolist())\n        if start % 40 == 0:\n            LOGGER.info("Reranking %d/%d", start + len(scores), len(payloads))\n    by_index = dict(zip(selected, values))\n    final = pd.Series(index=frame.index, dtype=float)\n    for indexes in ranked_groups:\n        head = indexes[:CONFIG["rerank_top_k"]]\n        reordered = sorted(head, key=by_index.__getitem__, reverse=True) + indexes[CONFIG["rerank_top_k"]:]\n        for rank, index in enumerate(reordered):\n            final.loc[index] = len(reordered) - rank\n    result = {"qual": ranking_metrics(frame, final.loc[frame.index].to_numpy(), "qual"),\n              "reranked_pairs": len(selected), "elapsed_seconds": time.monotonic() - started,\n              "scope": "test qualification labeled pools; semantic top10 prefix only"}\n    write_json(output / "reranker_audit.json", result)\n    pd.DataFrame({"pair_index": selected, "reranker_yes_score": values}).to_csv(output / "reranker_scores.csv", index=False)\n    del model\n    gc.collect()\n    torch.cuda.empty_cache()\n    return result\n\n\ndef write_report(results: dict, output: Path) -> None:\n    """Functionality: write the readable experiment report.\n    Inputs: metric results and output directory. Outputs: Markdown report.\n    Logic: report each task and candidate-pool metric separately.\n    Constraints: do not describe synthetic-label results as real hiring or interview gains."""\n    lines = ["# JobRec cold-job feasibility v1", "", "Completed on Kaggle; frozen Qwen embeddings + masked dual-head MLP.",\n             "", "| Model | Task | AP | AUC | NDCG@5 | Recall@5 | Queries | Median labeled pool |",\n             "|---|---|---:|---:|---:|---:|---:|---:|"]\n    for name, metrics in results["models"].items():\n        for task, row in metrics.items():\n            lines.append(f"| {name} | {task} | {row[\'average_precision\']:.4f} | {row[\'roc_auc\']:.4f} | "\n                         f"{row[\'ndcg@5\']:.4f} | {row[\'recall@5\']:.4f} | {row[\'positive_queries\']} | {row[\'median_labeled_pool\']:.1f} |")\n    lines += ["", "## Reranker", "", json.dumps(results["reranker"], indent=2), "", "## Limits", "",\n              "- Fully synthetic, rule-derived supervision; no real hiring outcome validation.",\n              "- Jobs split 70/15/15; candidates may recur. This is a new cold-job protocol, not paper reproduction.",\n              "- Source repeated labels collapsed only after consistency checks; unknown labels stay masked.",\n              "- Rankings use observed labeled pools, not the full catalog; both task pools differ.",\n              "- Qualification labels are conditional on the source selection process and are very sparse.",\n              "- No interview features exist in this dataset; interview benefit has not been tested.",\n              "- Text length, seed, checkpoints and training settings were fixed before execution.",\n              "- AP prevalence baselines and evaluated query counts are in metrics.json.",\n              "- Dataset repository license was not explicit; kernel remains private."]\n    (output / "RESULTS.md").write_text("\\n".join(lines) + "\\n", encoding="utf-8")\n\n\ndef main() -> None:\n    """Functionality: execute the full experiment.\n    Inputs: Kaggle GPU/network environment. Outputs: artifacts under `/kaggle/working/jobrec_v1`.\n    Logic: save audit data after each stage, then save models, predictions, and metrics.\n    Constraints: record the failing stage and re-raise any exception; do not rerun, degrade, or change\n    parameters automatically. Preserve evidence from completed stages."""\n    output = Path("/kaggle/working/jobrec_v1")\n    output.mkdir(parents=True, exist_ok=True)\n    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s",\n                        handlers=[logging.StreamHandler(sys.stdout), logging.FileHandler(output / "execution.log", encoding="utf-8")])\n    stage, started = "bootstrap", time.monotonic()\n    write_json(output / "config.json", CONFIG)\n    try:\n        bootstrap(output)\n        stage = "download_and_audit"\n        frames = download_data(output)\n        pairs, users, jobs, texts, structured = prepare_data(frames, output)\n        stage = "baselines"\n        lexical = lexical_baseline(pairs, texts, output)\n        logistic = structured_baseline(pairs, structured, output)\n        stage = "frozen_qwen_embedding"\n        embeddings = encode_profiles(texts, output)\n        features, cosine = build_pair_features(pairs, embeddings, structured, output)\n        stage = "dual_head_training"\n        neural = train_dual_head(pairs, features, output)\n        predictions = {"tfidf": lexical, "structured_logistic": logistic,\n                       "qwen_embedding": cosine, "dual_head_mlp": neural}\n        results = {"protocol": CONFIG["protocol"],\n                   "models": {name: evaluate_scores(pairs, score, "test") for name, score in predictions.items()}}\n        write_json(output / "metrics_before_reranker.json", results)\n        for name, values in predictions.items():\n            pairs[name + "_pref"] = values[:, 0]\n            pairs[name + "_qual"] = values[:, 1]\n        pairs.loc[pairs.split.eq("test")].to_csv(output / "test_predictions.csv", index=False)\n        stage = "qualification_reranker"\n        results["reranker"] = rerank_qualification(pairs, texts, cosine, output)\n        results["total_seconds"] = time.monotonic() - started\n        write_json(output / "metrics.json", results)\n        write_report(results, output)\n        write_json(output / "status.json", {"status": "complete", "elapsed_seconds": results["total_seconds"]})\n        LOGGER.info("EXPERIMENT_COMPLETE %s", json.dumps(results))\n    except Exception as error:\n        write_json(output / "status.json", {"status": "failed", "stage": stage,\n                   "error_type": type(error).__name__, "error": str(error), "elapsed_seconds": time.monotonic() - started})\n        LOGGER.exception("Experiment failed at stage=%s; no automatic fallback", stage)\n        raise\n\n\nif __name__ == "__main__":\n    main()\n'
module = types.ModuleType('jobrec_v1')
exec(compile(source, 'jobrec_v1.py', 'exec'), module.__dict__)
sys.modules['jobrec_v1'] = module


In [ ]:
"""Responsibilities: register a frozen experiment module.
Implementation: compile source in an isolated module namespace.
Related Modules: later notebook cells import this registered module.
Declaration Index: None.
Variable Index: source is embedded Python; module is the registered namespace.
"""
import sys
import types
source = '"""Responsibilities: measure feature and task-sharing contributions using fixed v1 data and embeddings.\nImplementation: verify input hashes, compare four networks from the same initialization, and use paired\nquery bootstrap on test predictions.\nRelated Modules: jobrec_v1 is the original experiment.py embedded in the notebook; its source hash is fixed.\n\nDeclaration Index:\n- verify_inputs: verify parent artifact identity.\n- feature_view: remove the registered feature group for one comparison variant.\n- IndependentHeads: define separate task networks initialized from the shared starting model.\n- IndependentHeads.__init__: clone the shared trunk and each task\'s corresponding head.\n- IndependentHeads.forward: return both independent task logits.\n- masked_loss: compute loss only for observed labels.\n- train_variant: run one controlled training comparison and score every pair.\n- bootstrap_ap: compute paired AP under query-resampling counts.\n- query_bootstrap: compute model intervals and preregistered paired differences.\n- main: validate inputs, train four comparisons, evaluate, and save audit results.\n\nVariable Index:\n- CONFIG: inherited fixed v1 experiment configuration.\n- PROTOCOL: new v2 ablation conditions.\n- INPUT_HASHES: hashes binding the required parent inputs.\n- LOGGER: stage-level logs without credentials or profile text.\n- branches: instance state containing two networks with independent parameters.\n\nConstraints: do not modify v1, split, seed, or hyperparameters; bootstrap intervals do not test\ntraining-seed variability."""\n\nfrom __future__ import annotations\n\nimport copy\nimport gc\nimport hashlib\nimport json\nimport logging\nimport random\nimport sys\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom sklearn.metrics import average_precision_score\n\nimport jobrec_v1 as base\n\nCONFIG = base.CONFIG\nPROTOCOL = {\n    "name": "jobrec-fixed-split-ablation-v2",\n    "variants": ["shared_full", "shared_structured", "shared_vector", "independent_full"],\n    "bootstrap_replicates": 2000,\n    "bootstrap_seed": 42,\n    "initialization": "reset seed42 immediately before every variant; clone identical shared trunk for independent branches",\n    "feature_ablation": "zero removed blocks, retaining 2059 inputs and identical shared parameter counts",\n    "checkpoint_selection": "same joint mean validation AP for all four variants",\n    "comparison_scope": "v2-to-v2 controlled comparison; v1 RNG position before network initialization may differ",\n}\nINPUT_HASHES = {\n    "config.json": "8dea69e2e61e7b5ce0812072f04eb17f931aad0c2fa97aeba2deb88e8db28513",\n    "data_manifest.json": "9f692a0f8921625287408c9cbd5343823a2c612e50a43db0937667b6d0c73957",\n    "pair_splits.csv": "37b134a5a2461481ce9d04318c6d05bf5025752173236b334db97cf130db63c0",\n    "split_manifest.json": "dd228b8394d9d14e0bca0c77fadb07625d7f7ff3ee07472a2a93ed083cfd7eb1",\n    "embeddings.npz": "c468440b592455f1503b022784c7ab44a7c4b9600ffe4a17e3f46f462ec2cc3b",\n    "test_predictions.csv": "2045a94220c1d61bb19a3de4da26d10fe74eab5853d048e9eeb3bbb23b0b660b",\n    "feature_scaler.json": "4ec782a0009581b88c7bcd21d824c410181350916cbda163b1680b31f53a5051",\n}\nLOGGER = logging.getLogger("jobrec_ablation")\n\n\ndef verify_inputs(folder: Path) -> None:\n    """Functionality: verify the identity of parent experiment artifacts.\n    Inputs: parent output directory. Outputs: none.\n    Logic: compare SHA256 values for each required file and validate configuration.\n    Constraints: reject updated parent artifacts even when filenames are unchanged."""\n    for name, expected in INPUT_HASHES.items():\n        actual = hashlib.sha256((folder / name).read_bytes()).hexdigest()\n        if actual != expected:\n            raise ValueError(f"Parent artifact changed: {name}: {actual}")\n    if json.loads((folder / "config.json").read_text()) != CONFIG:\n        raise ValueError("Parent config differs from embedded v1 source")\n    LOGGER.info("Verified all %d parent artifact hashes", len(INPUT_HASHES))\n\n\ndef feature_view(features: np.ndarray, variant: str) -> np.ndarray:\n    """Functionality: remove one feature group for a registered comparison.\n    Inputs: N-by-2059 matrix and experiment name. Outputs: an independent copy.\n    Logic: zero the first 2048 columns for the structured-only view or the final 11 columns for the\n    embedding-only view.\n    Constraints: preserve the source matrix. Layer widths and parameter counts remain the same, but\n    zeroed inputs reduce the effective usable feature capacity."""\n    if features.shape[1] != 2059 or variant not in PROTOCOL["variants"]:\n        raise ValueError("Unexpected feature dimension or variant")\n    result = features.copy()\n    if variant == "shared_structured":\n        result[:, :-11] = 0\n    elif variant == "shared_vector":\n        result[:, -11:] = 0\n    return result\n\n\nclass IndependentHeads(torch.nn.Module):\n    """Functionality: define independent task networks.\n    Logic: each network copies the shared initial trunk and its corresponding task head.\n    Constraints: preserve each task\'s layer width, approximately double total parameters, and interpret\n    this as a sharing comparison rather than an equal-parameter comparison."""\n\n    def __init__(self, template: base.DualHeadModel):\n        """Functionality: create independent task branches.\n        Inputs: initialized shared model. Outputs: a new IndependentHeads instance.\n        Logic: deepcopy prevents parameter aliasing and does not consume new initialization random numbers.\n        Constraints: only `branches` carries model state."""\n        super().__init__()\n        self.branches = torch.nn.ModuleList([\n            torch.nn.Sequential(copy.deepcopy(template.trunk), copy.deepcopy(head))\n            for head in template.heads\n        ])\n\n    def forward(self, features: torch.Tensor) -> torch.Tensor:\n        """Functionality: compute logits from the independent task branches.\n        Inputs: B-by-2059 tensor. Outputs: B-by-2 logits.\n        Logic: concatenate each branch\'s task output.\n        Constraints: dropout masks are sampled independently in the two branches during training."""\n        return torch.cat([branch(features) for branch in self.branches], dim=1)\n\n\ndef masked_loss(logits: torch.Tensor, labels: torch.Tensor, observed: torch.Tensor,\n                positive_weights: torch.Tensor) -> torch.Tensor:\n    """Functionality: calculate loss over observed labels only.\n    Inputs: B-by-2 logits, labels, mask, and two positive-class weights. Outputs: scalar loss.\n    Logic: average within each task\'s observed labels, then average across tasks, matching v1.\n    Constraints: unknown positions must contain finite placeholder values; raise explicitly if neither task\n    has an observed label."""\n    element = torch.nn.functional.binary_cross_entropy_with_logits(\n        logits, labels, pos_weight=positive_weights, reduction="none")\n    losses = [(element[:, c] * observed[:, c]).sum() / observed[:, c].sum()\n              for c in range(2) if observed[:, c].any()]\n    if not losses:\n        raise ValueError("Batch has no observed task labels")\n    return torch.stack(losses).mean()\n\n\ndef train_variant(pairs: pd.DataFrame, features: np.ndarray, variant: str, output: Path) -> np.ndarray:\n    """Functionality: run one controlled network comparison.\n    Inputs: pairs, feature matrix, variant name, and output directory. Outputs: N-by-2 scores.\n    Logic: reset the same seed before each variant; retain minibatch order, loss, and joint validation\n    selection; save the selected checkpoint.\n    Constraints: select only within 12 epochs and never use test labels; v1\'s pretraining RNG position is\n    not treated as part of this comparison."""\n    random.seed(CONFIG["seed"])\n    np.random.seed(CONFIG["seed"])\n    torch.manual_seed(CONFIG["seed"])\n    torch.cuda.manual_seed_all(CONFIG["seed"])\n    template = base.DualHeadModel(features.shape[1])\n    model = (IndependentHeads(template) if variant == "independent_full" else template).to("cuda:0")\n    parameter_count = sum(p.numel() for p in model.parameters())\n    initial = {k: value.cpu() for k, value in template.state_dict().items()}\n    torch.save(initial, output / "initial_shared_template.pt")\n    x = torch.from_numpy(feature_view(features, variant)).to("cuda:0")\n    targets = pairs[["label_pref", "label_qual"]].to_numpy(dtype=np.float32)\n    observed = np.isfinite(targets)\n    y = torch.from_numpy(np.nan_to_num(targets, nan=0)).to("cuda:0")\n    mask = torch.from_numpy(observed).to("cuda:0")\n    train_ids = np.flatnonzero(pairs.split.eq("train"))\n    val_ids = np.flatnonzero(pairs.split.eq("validation"))\n    weights = []\n    for column in range(2):\n        labels = targets[train_ids, column]\n        labels = labels[np.isfinite(labels)]\n        weights.append(float((labels == 0).sum() / (labels == 1).sum()))\n    positive_weights = torch.tensor(weights, device="cuda:0")\n    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["learning_rate"], weight_decay=CONFIG["weight_decay"])\n    best_score, best_state, history = -1.0, None, []\n    rng = np.random.default_rng(CONFIG["seed"])\n    started = time.monotonic()\n    for epoch in range(1, CONFIG["epochs"] + 1):\n        model.train()\n        losses, order = [], rng.permutation(train_ids)\n        for start in range(0, len(order), CONFIG["training_batch_size"]):\n            ids = order[start:start + CONFIG["training_batch_size"]]\n            optimizer.zero_grad(set_to_none=True)\n            loss = masked_loss(model(x[ids]), y[ids], mask[ids], positive_weights)\n            loss.backward()\n            optimizer.step()\n            losses.append(float(loss.item()))\n        model.eval()\n        with torch.inference_mode():\n            val_scores = torch.sigmoid(model(x[val_ids])).cpu().numpy()\n        aps = [float(average_precision_score(targets[val_ids, c][observed[val_ids, c]],\n               val_scores[:, c][observed[val_ids, c]])) for c in range(2)]\n        validation_mean = float(np.mean(aps))\n        if not np.isfinite(validation_mean):\n            raise ValueError("Non-finite validation AP")\n        row = {"epoch": epoch, "mean_batch_loss": float(np.mean(losses)), "validation_ap": aps}\n        history.append(row)\n        LOGGER.info("%s epoch=%s", variant, row)\n        if validation_mean > best_score:\n            best_score, best_state, selected_epoch = validation_mean, copy.deepcopy(model.state_dict()), epoch\n    model.load_state_dict(best_state)\n    with torch.inference_mode():\n        scores = torch.sigmoid(model(x)).cpu().numpy()\n    if not np.isfinite(scores).all():\n        raise ValueError("Non-finite prediction")\n    torch.save({"state_dict": {k: v.cpu() for k, v in best_state.items()}, "variant": variant,\n                "config": CONFIG, "protocol": PROTOCOL}, output / "checkpoint.pt")\n    base.write_json(output / "training_history.json", {"epochs": history, "selected_epoch": selected_epoch,\n                    "validation_mean_ap": best_score, "parameter_count": parameter_count,\n                    "positive_weights": weights, "training_seconds": time.monotonic() - started})\n    del model, template, x, y, mask, optimizer, best_state\n    gc.collect()\n    torch.cuda.empty_cache()\n    return scores\n\n\ndef bootstrap_ap(labels: np.ndarray, scores: np.ndarray, group_ids: np.ndarray,\n                 counts: np.ndarray) -> np.ndarray:\n    """Functionality: compute paired AP for query-resampled test sets.\n    Inputs: labels, scores, group indices, and bootstrap query counts. Outputs: one AP per replicate.\n    Logic: use query multiplicities as sample weights and compute precision steps across tied scores.\n    Constraints: this is not mean query AP; reject replicates without positives; process 100 replicates per\n    block to limit memory."""\n    order = np.argsort(-scores, kind="stable")\n    sorted_labels, sorted_scores = labels[order], scores[order]\n    ends = np.r_[np.flatnonzero(np.diff(sorted_scores)), len(scores) - 1]\n    values = []\n    for start in range(0, len(counts), 100):\n        weights = counts[start:start + 100, group_ids[order]].astype(float)\n        positive = np.cumsum(weights * sorted_labels, axis=1)[:, ends]\n        total = np.cumsum(weights, axis=1)[:, ends]\n        if (positive[:, -1] == 0).any():\n            raise ValueError("Bootstrap replicate has no positives")\n        precision = np.divide(positive, total, out=np.zeros_like(positive), where=total > 0)\n        increments = np.diff(np.column_stack([np.zeros(len(positive)), positive]), axis=1)\n        values.extend(((increments * precision).sum(axis=1) / positive[:, -1]).tolist())\n    return np.asarray(values)\n\n\ndef query_bootstrap(predictions: pd.DataFrame, model_names: list[str], output: Path) -> dict:\n    """Functionality: quantify query-sampling uncertainty for fixed models.\n    Inputs: test predictions, model names, and output directory. Outputs: interval reports.\n    Logic: sample preference candidates and qualification jobs 2,000 times; reuse the same draws for all\n    models within each task; report AP, nDCG@5, Recall@5, and preregistered paired differences.\n    Constraints: percentile 95% intervals are exploratory and unadjusted for multiple comparisons.\n    Preference results are conditional on the current test jobs and do not include training-seed or\n    alternative-split variation. Exclude all-negative queries only from ranking means."""\n    result = {"replicates": PROTOCOL["bootstrap_replicates"], "models": {}, "comparisons": {}}\n    contrasts = [("shared_full", name) for name in (\n        "shared_structured", "shared_vector", "independent_full", "structured_logistic")]\n    for task in ("pref", "qual"):\n        frame = predictions.loc[predictions["label_" + task].notna()].reset_index(drop=True)\n        group_column = "user" if task == "pref" else "item"\n        groups, ids = np.unique(frame[group_column], return_inverse=True)\n        labels = frame["label_" + task].to_numpy()\n        rng = np.random.default_rng(PROTOCOL["bootstrap_seed"])\n        counts = rng.multinomial(len(groups), np.full(len(groups), 1 / len(groups)),\n                                 size=PROTOCOL["bootstrap_replicates"])\n        distributions = {}\n        for name in model_names:\n            scores = frame[name + "_" + task].to_numpy()\n            ap = bootstrap_ap(labels, scores, ids, counts)\n            query_scores = np.full((len(groups), 2), np.nan)\n            for i in range(len(groups)):\n                group = frame.loc[ids == i]\n                if group["label_" + task].sum() > 0:\n                    metrics = base.ranking_metrics(group, scores[ids == i], task)\n                    query_scores[i] = [metrics["ndcg@5"], metrics["recall@5"]]\n            valid = np.isfinite(query_scores[:, 0])\n            denominator = counts[:, valid].sum(axis=1)\n            if (denominator == 0).any():\n                raise ValueError("Bootstrap replicate has no positive queries")\n            rank = counts[:, valid] @ query_scores[valid] / denominator[:, None]\n            distributions[name] = {"average_precision": ap, "ndcg@5": rank[:, 0], "recall@5": rank[:, 1]}\n            result["models"].setdefault(name, {})[task] = {\n                metric: np.quantile(values, [0.025, 0.975]).tolist()\n                for metric, values in distributions[name].items()\n            }\n            LOGGER.info("Bootstrap completed task=%s model=%s queries=%d", task, name, len(groups))\n        for left, right in contrasts:\n            key = left + "_minus_" + right\n            result["comparisons"].setdefault(key, {})[task] = {\n                metric: {"ci95": np.quantile(distributions[left][metric] - distributions[right][metric],\n                                             [0.025, 0.975]).tolist()}\n                for metric in distributions[left]\n            }\n    base.write_json(output / "bootstrap.json", result)\n    return result\n\n\ndef main() -> None:\n    """Functionality: run the ablation and result audit.\n    Inputs: mounted v1 artifacts and Kaggle GPU environment. Outputs: `ablation_v2` artifacts.\n    Logic: verify parent identity, reconstruct pairs/features, train four variants, evaluate, and analyze\n    uncertainty.\n    Constraints: record failures and re-raise; do not change hyperparameters, retry, or fall back. Reuse\n    verified embeddings rather than downloading or encoding them again."""\n    output = Path("/kaggle/working/ablation_v2")\n    output.mkdir(parents=True, exist_ok=True)\n    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s",\n                        handlers=[logging.StreamHandler(sys.stdout), logging.FileHandler(output / "execution.log")])\n    stage, started = "parent_verification", time.monotonic()\n    base.write_json(output / "protocol.json", {"base_config": CONFIG, "ablation": PROTOCOL, "input_hashes": INPUT_HASHES})\n    try:\n        candidates = list(Path("/kaggle/input").rglob("jobrec_v1/config.json"))\n        if len(candidates) != 1:\n            raise ValueError(f"Expected one mounted parent artifact, found {len(candidates)}")\n        parent = candidates[0].parent\n        verify_inputs(parent)\n        stage = "environment_and_data"\n        base.bootstrap(output)\n        frames = base.download_data(output)\n        if json.loads((output / "data_manifest.json").read_text()) != json.loads((parent / "data_manifest.json").read_text()):\n            raise ValueError("Downloaded source data changed")\n        pairs, _, _, _, structured = base.prepare_data(frames, output)\n        pd.testing.assert_frame_equal(pairs, pd.read_csv(parent / "pair_splits.csv"), check_dtype=False)\n        archived = np.load(parent / "embeddings.npz")\n        features, _ = base.build_pair_features(pairs, (archived["candidates"], archived["jobs"]), structured, output)\n        if json.loads((output / "feature_scaler.json").read_text()) != json.loads((parent / "feature_scaler.json").read_text()):\n            raise ValueError("Feature preprocessing differs from v1")\n        results, scores = {}, {}\n        for variant in PROTOCOL["variants"]:\n            stage = "training_" + variant\n            destination = output / variant\n            destination.mkdir()\n            scores[variant] = train_variant(pairs, features, variant, destination)\n        stage = "evaluation"\n        test = pairs.loc[pairs.split.eq("test")].copy()\n        for name, values in scores.items():\n            results[name] = base.evaluate_scores(pairs, values, "test")\n            test[name + "_pref"] = values[test.index, 0]\n            test[name + "_qual"] = values[test.index, 1]\n        previous = pd.read_csv(parent / "test_predictions.csv")\n        pd.testing.assert_frame_equal(test[["user", "item"]].reset_index(drop=True), previous[["user", "item"]])\n        for task in ("pref", "qual"):\n            test["structured_logistic_" + task] = previous["structured_logistic_" + task].to_numpy()\n        results["structured_logistic"] = base.evaluate_scores(test,\n            test[["structured_logistic_pref", "structured_logistic_qual"]].to_numpy(), "test")\n        test.to_csv(output / "test_predictions.csv", index=False)\n        base.write_json(output / "metrics.json", results)\n        stage = "query_bootstrap"\n        query_bootstrap(test, list(results), output)\n        base.write_json(output / "status.json", {"status": "complete", "total_seconds": time.monotonic() - started})\n        LOGGER.info("ABLATION_COMPLETE %s", json.dumps(results))\n    except Exception as error:\n        base.write_json(output / "status.json", {"status": "failed", "stage": stage,\n                        "error_type": type(error).__name__, "error": str(error)})\n        LOGGER.exception("Ablation failed at stage=%s; no fallback", stage)\n        raise\n\n\nif __name__ == "__main__":\n    main()\n'
module = types.ModuleType('jobrec_v2')
exec(compile(source, 'jobrec_v2.py', 'exec'), module.__dict__)
sys.modules['jobrec_v2'] = module


In [ ]:
"""Responsibilities: register a frozen experiment module.
Implementation: compile source in an isolated module namespace.
Related Modules: later notebook cells import this registered module.
Declaration Index: None.
Variable Index: source is embedded Python; module is the registered namespace.
"""
import sys
import types
source = '"""Responsibilities: define the frozen v4 protocol, entity-level profile masking, and missing-aware features.\nImplementation: derive deterministic masks from SHA256, hide source fields before feature calculation,\nthen compute 11 NaN-aware pairwise features.\nRelated Modules: jobrec_v1 supplies the established list parser and feature names; experiment trains;\njobrec_missing_eval evaluates.\n\nDeclaration Index:\n- uniform: produce a deterministic uniform value from the registered identity tuple.\n- scenarios: list the fixed evaluation scenarios and repeat counts.\n- make_masks: create deterministic user/job field masks without using labels.\n- hide_profiles: physically hide selected source values before feature construction.\n- unknown: classify a scalar as missing without conflating invalid values with absence.\n- number: parse known numeric values while preserving missingness.\n- sequence: parse known list fields while preserving unknown versus empty.\n- academic: map a supported academic-level code.\n- boolean: convert a supported summer-commitment value.\n- features: recompute the 11-dimensional pair feature matrix from visible source fields.\n- feature_view: build the registered 11-, 22-, or 6-column model input.\n\nVariable Index:\n- USER_BLOCKS/JOB_BLOCKS: atomic user/job profile groups used for masking.\n- OPTIONAL/CORE: fixed feature-column indices for the registered variants.\n- LEVELS: supported academic-level encoding.\n- RANKER_CONFIG: fixed historical tree parameters.\n- PROTOCOL: approved v4 scenarios, model variants, and selection conditions.\n- HISTORICAL: hashes for the v3 reference rankers and predictions.\n\nConstraints: missingness is not chosen from labels; do not zero-fill or infer hidden values; do not\nretry. IDs are used to derive masks but are not prediction features."""\n\nimport hashlib\nimport json\n\nimport numpy as np\nimport pandas as pd\n\nimport jobrec_v1 as base\n\nUSER_BLOCKS = [("skills",), ("interests",), ("masters_major", "second_major", "undergrad_major"),\n               ("in_person_commitment",), ("gpa",), ("months_experience",), ("academic_level",),\n               ("hours_per_week",), ("length_of_commitment",), ("num_publications",), ("commit_to_summer",)]\nJOB_BLOCKS = [(x,) for x in ("required_skills", "industry", "acceptable_majors", "job_in_person_commitment",\n              "min_gpa", "min_months_experience", "min_academic_level", "min_hours_per_week",\n              "min_length_of_commitment", "min_num_publications")]\nOPTIONAL = [4, 7, 8, 9, 10]\nCORE = [0, 1, 2, 3, 5, 6]\nLEVELS = {"UG1": 1, "UG2": 2, "UG3": 3, "UG4": 4, "MS1": 5, "MS2": 6,\n          "PhD1": 7, "PhD2": 8, "PhD3": 9, "PhD4": 10, "PhD5": 11}\nRANKER_CONFIG = {"objective": "lambdarank", "n_estimators": 200, "learning_rate": 0.05,\n                 "num_leaves": 15, "min_child_samples": 20, "reg_lambda": 1.0,\n                 "random_state": 42, "n_jobs": 2, "verbosity": -1,\n                 "deterministic": True, "force_col_wise": True}\nPROTOCOL = {"name": "jobrec-missing-v4", "seed": 42, "mask_replicates": 5,\n            "training_mix": {"clean": 0.5, "random_30": 0.25, "optional_bundle": 0.25},\n            "validation_scenarios": ["clean", "random_30", "optional_bundle"],\n            "primary_scenarios": ["random_30", "optional_bundle", "mixed_optional"],\n            "selection": "per-task equal mean validation AP; grid20..200; earliest tie",\n            "models": ["H", "A", "B", "C", "D"], "tree_grid": list(range(20, 201, 20)),\n            "bootstrap_replicates": 2000, "clean_max_drop": 0.02, "missing_min_gain": 0.03,\n            "ranker": RANKER_CONFIG, "user_blocks": USER_BLOCKS, "job_blocks": JOB_BLOCKS,\n            "optional_columns": OPTIONAL, "core_columns": CORE,\n            "feature_names": list(base.STRUCTURED_NAMES), "lightgbm": "4.6.0",\n            "scope": "development diagnostics; synthetic labels; no interview fusion or deployment"}\nHISTORICAL = {\n    "lambdarank/pref.txt": "7cf96ed020d367e9798893ffe7713674620073bda0941a7e1e8682758cb2d8cb",\n    "lambdarank/qual.txt": "b0c310f4dcb66c375794fbe2f931a19ca108ab17709d64ef0fc6d0daca9b00e7",\n    "test_predictions.csv": "1eff7faac87ec5011f70c247898935d7569169d3dcc28337fc3d1aa5d4431f62",\n    "metrics.json": "577b26e0b871adbbf7f3e98806b5e862a805e3d95178c437be28ff6b7a676a2d",\n}\n\n\ndef uniform(namespace: str, scenario: str, repeat: int, entity: str, block: str) -> float:\n    """Functionality: return a stable pseudo-random value in [0, 1).\n    Inputs: namespace, scenario, repeat number, entity ID, and profile block.\n    Outputs: deterministic float in [0, 1).\n    Logic: serialize the fixed seed and identity tuple as compact JSON, hash it with SHA256, and map the\n    first eight bytes to an unsigned integer fraction.\n    Constraints: independent of row order and Python process hashing."""\n    raw = json.dumps([42, namespace, scenario, repeat, str(entity), block], separators=(",", ":")).encode()\n    return int.from_bytes(hashlib.sha256(raw).digest()[:8], "big") / 2**64\n\n\ndef scenarios() -> list[tuple[str, int]]:\n    """Functionality: enumerate the fixed evaluation scenarios.\n    Inputs: the module-level frozen protocol. Outputs: ordered scenario/repeat pairs.\n    Logic: include each deterministic scenario once and five random scenarios five times each.\n    Constraints: repeat zero is the primary report; repeats are not selected based on scores."""\n    fixed = ["clean", "optional_bundle", "skills_only", "no_candidate_information"]\n    fixed += ["single_" + str(i) for i in range(11)]\n    random = ["random_10", "random_30", "random_50", "mixed_optional", "job_random_30"]\n    return [(name, 0) for name in fixed] + [(name, r) for name in random for r in range(5)]\n\n\ndef make_masks(users: pd.DataFrame, jobs: pd.DataFrame, scenario: str, repeat: int = 0,\n               namespace: str = "evaluation") -> tuple[np.ndarray, np.ndarray, list[str]]:\n    """Functionality: create user and job profile masks for a registered scenario.\n    Inputs: entity tables with IDs, scenario, repeat index, and mask namespace.\n    Outputs: user mask, job mask, and user assignment labels.\n    Logic: select fixed blocks or hash-derived blocks; assign training-mix users to the 50/25/25 groups.\n    Constraints: only IDs determine assignments; labels are never read."""\n    allowed = {name for name, _ in scenarios()} | {"training_mix"}\n    if scenario not in allowed or repeat not in range(5):\n        raise ValueError("Unregistered mask scenario or repeat")\n    um = np.zeros((len(users), 11), dtype=bool)\n    jm = np.zeros((len(jobs), 10), dtype=bool)\n    assignments = []\n    for row, entity in enumerate(users.student_id):\n        mode = scenario\n        if scenario == "training_mix":\n            choice = uniform(namespace, scenario, repeat, entity, "assignment")\n            mode = "clean" if choice < .5 else "random_30" if choice < .75 else "optional_bundle"\n        assignments.append(mode)\n        if mode.startswith("single_"):\n            um[row, int(mode.split("_")[1])] = True\n        elif mode == "optional_bundle":\n            um[row, OPTIONAL] = True\n        elif mode == "mixed_optional":\n            if uniform(namespace, scenario, repeat, entity, "assignment") < .5:\n                um[row, OPTIONAL] = True\n        elif mode == "skills_only":\n            um[row, 1:] = True\n        elif mode == "no_candidate_information":\n            um[row] = True\n        elif mode.startswith("random_"):\n            probability = int(mode.split("_")[1]) / 100\n            um[row] = [uniform(namespace, scenario, repeat, entity, "/".join(block)) < probability\n                       for block in USER_BLOCKS]\n    if scenario == "job_random_30":\n        for row, entity in enumerate(jobs.job_id):\n            jm[row] = [uniform(namespace, scenario, repeat, entity, "/".join(block)) < .3\n                       for block in JOB_BLOCKS]\n    return um, jm, assignments\n\n\ndef hide_profiles(frame: pd.DataFrame, masks: np.ndarray, blocks: list) -> pd.DataFrame:\n    """Functionality: create copies of profile tables with selected source fields hidden.\n    Inputs: entity table, Boolean block mask, and block definitions.\n    Outputs: an independent table with selected values set to None.\n    Logic: cast affected columns to object before replacing selected cells.\n    Constraints: never mutate the source table or retain hidden values for feature calculation."""\n    if masks.shape != (len(frame), len(blocks)):\n        raise ValueError("Mask shape differs from entity table")\n    result = frame.copy()\n    for col, block in enumerate(blocks):\n        for name in block:\n            result[name] = result[name].astype(object)\n            result.loc[masks[:, col], name] = None\n    return result\n\n\ndef unknown(value: object) -> bool:\n    """Functionality: determine whether a scalar denotes unknown data.\n    Inputs: one field value. Outputs: Boolean.\n    Logic: only None and floating-point NaN are unknown.\n    Constraints: empty strings, infinities, and invalid types are not converted to missing; list fields use\n    sequence parsing."""\n    return value is None or isinstance(value, (float, np.floating)) and np.isnan(value)\n\n\ndef number(value: object) -> float:\n    """Functionality: parse one numeric profile field while preserving absence.\n    Inputs: one scalar. Outputs: a finite float or NaN.\n    Logic: retain values classified as unknown; convert supported known values to float.\n    Constraints: reject infinity, Boolean values, and invalid text; preserve a known zero."""\n    if unknown(value):\n        return np.nan\n    if isinstance(value, (bool, np.bool_)):\n        raise ValueError("Boolean is not a numeric quantity")\n    result = float(value)\n    if not np.isfinite(result):\n        raise ValueError("Invalid non-finite numeric quantity")\n    return result\n\n\ndef sequence(value: object) -> set | None:\n    """Functionality: parse a skill, interest, or major list while preserving unknown state.\n    Inputs: one field value. Outputs: a set or None.\n    Logic: use the established v1 parser for known values.\n    Constraints: do not convert unknown to an empty set or alter case."""\n    return None if unknown(value) else set(base.parse_list(value))\n\n\ndef academic(value: object) -> float:\n    """Functionality: validate and map an academic-level code.\n    Inputs: one code. Outputs: the supported original code or NaN.\n    Logic: look up the fixed LEVELS mapping.\n    Constraints: reject unsupported codes rather than treating them as ordinary missing data."""\n    return np.nan if unknown(value) else float(LEVELS[value])\n\n\ndef boolean(value: object) -> float:\n    """Functionality: convert the summer-commitment field.\n    Inputs: one field value. Outputs: 0, 1, or NaN.\n    Logic: accept only Boolean values or the registered true/false strings.\n    Constraints: unknown does not mean false; invalid strings raise an error; valid source meaning is\n    unchanged."""\n    if unknown(value):\n        return np.nan\n    text = str(value).lower()\n    if text not in ("true", "false"):\n        raise ValueError("Invalid commitment boolean")\n    return float(text == "true")\n\n\ndef features(pairs: pd.DataFrame, users: pd.DataFrame, jobs: pd.DataFrame) -> np.ndarray:\n    """Functionality: recompute missing-aware pair features from masked source profiles.\n    Inputs: pair-index table and the already-hidden user/job tables.\n    Outputs: an N-by-11 float32 matrix.\n    Logic: parse each entity once, compute set matches per pair, and vectorize numeric margins.\n    Constraints: do not access a complete-profile cache. A partially known major set remains usable; only\n    an entirely unknown major block yields NaN. Indices must refer to the supplied entity tables. An empty\n    job-skill requirement preserves the historical error. NaN is allowed; infinity is rejected."""\n    ui, ji = pairs.u_idx.to_numpy(), pairs.j_idx.to_numpy()\n    us = [sequence(v) for v in users.skills]\n    req = [sequence(v) for v in jobs.required_skills]\n    if any(v is not None and not v for v in req):\n        raise ValueError("Empty required skill list is outside the frozen protocol")\n    interests = [sequence(v) for v in users.interests]\n    majors = [{v for v in row if not unknown(v)} or None\n              for row in users[["masters_major", "second_major", "undergrad_major"]].itertuples(index=False, name=None)]\n    accepted = [sequence(v) for v in jobs.acceptable_majors]\n    industries = jobs.industry.tolist()\n    modes_u, modes_j = users.in_person_commitment.tolist(), jobs.job_in_person_commitment.tolist()\n    result = np.full((len(pairs), 11), np.nan, dtype=np.float32)\n    for k, (u, j) in enumerate(zip(ui, ji)):\n        if us[u] is not None and req[j] is not None:\n            result[k, 0] = len(us[u] & req[j]) / len(req[j])\n        if interests[u] is not None and not unknown(industries[j]):\n            result[k, 1] = float(industries[j] in interests[u])\n        if majors[u] is not None and accepted[j] is not None:\n            result[k, 2] = float(bool(majors[u] & accepted[j]))\n        if not unknown(modes_u[u]) and not unknown(modes_j[j]):\n            result[k, 3] = float(modes_u[u] == modes_j[j])\n    for c in range(4, 10):\n        parser = academic if c == 6 else number\n        uv = np.asarray([parser(v) for v in users[USER_BLOCKS[c][0]]])\n        jv = np.asarray([parser(v) for v in jobs[JOB_BLOCKS[c][0]]])\n        result[:, c] = uv[ui] - jv[ji]\n    result[:, 10] = np.asarray([boolean(v) for v in users.commit_to_summer])[ui]\n    if np.isinf(result).any():\n        raise ValueError("Infinite feature")\n    return result\n\n\ndef feature_view(matrix: np.ndarray, model: str) -> np.ndarray:\n    """Functionality: construct the input view for a registered model variant.\n    Inputs: an N-by-11 feature array and variant name. Outputs: an 11-, 22-, or 6-column array.\n    Logic: C appends NaN indicators, D selects the fixed six columns, and other variants copy the input.\n    Constraints: reject unknown variants, infinity, and invalid dimensions."""\n    if model not in PROTOCOL["models"] or matrix.ndim != 2 or matrix.shape[1] != 11 or np.isinf(matrix).any():\n        raise ValueError("Invalid model or structured feature matrix")\n    if model == "C":\n        return np.concatenate([matrix, np.isnan(matrix).astype(np.float32)], axis=1)\n    return matrix[:, CORE].copy() if model == "D" else matrix.copy()\n'
module = types.ModuleType('jobrec_missing')
exec(compile(source, 'jobrec_missing.py', 'exec'), module.__dict__)
sys.modules['jobrec_missing'] = module


In [ ]:
"""Responsibilities: register a frozen experiment module.
Implementation: compile source in an isolated module namespace.
Related Modules: later notebook cells import this registered module.
Declaration Index: None.
Variable Index: source is embedded Python; module is the registered namespace.
"""
import sys
import types
source = '"""Responsibilities: evaluate fixed v4 predictions and save query-level evidence, intervals, and\nengineering candidate-gate results.\nImplementation: reuse v1 primary metrics, record ties and positive ranks explicitly, and reuse v2\'s\nweighted AP for paired query bootstrap intervals.\nRelated Modules: jobrec_missing defines the protocol; experiment calls this evaluator after training.\n\nDeclaration Index:\n- query_details: compute stable query rankings and positive-item positions.\n- summarize_strata: summarize positive-pair changes by available-feature count.\n- analyze: evaluate all fixed model/scenario predictions and write reports.\n\nVariable Index:\n- LOGGER: stage-level evaluation logging.\n- CONTRASTS: preregistered model-score differences.\n- Instance state: None.\n\nConstraints: share bootstrap draws across models/scenarios within each task; exclude all-negative\nqueries only from ranking means; diagnostics do not establish fairness."""\n\nimport logging\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nimport jobrec_v1 as base\nimport jobrec_v2 as previous\nimport jobrec_missing as spec\n\nLOGGER = logging.getLogger("jobrec_v4_evaluation")\nCONTRASTS = [("B", "A"), ("C", "B"), ("D", "B")] + [(m, "H") for m in "ABCD"]\n\n\ndef query_details(frame: pd.DataFrame, scores: np.ndarray, task: str,\n                  available: np.ndarray) -> tuple[pd.DataFrame, pd.DataFrame]:\n    """Functionality: record per-query rankings and positive-item positions.\n    Inputs: observed rows for one task, their scores, task name, and available-feature counts.\n    Outputs: query summary table and positive-pair table.\n    Logic: use stable descending sort to preserve the historical tie rule and apply the same top-k limits.\n    Constraints: report tie rates; retain all-negative queries with NaN ranking metrics; ranks start at one\n    and depend on input order for tied scores."""\n    frame = frame.reset_index(drop=True)\n    rows, positives = [], []\n    for query, indices in frame.groupby("user" if task == "pref" else "item", sort=True).indices.items():\n        indices = np.asarray(indices)\n        labels = frame.iloc[indices]["label_" + task].to_numpy()\n        values = scores[indices]\n        order = np.argsort(-values, kind="stable")\n        ranked = labels[order]\n        count = int(labels.sum())\n        row = {"query": query, "pool": len(indices), "positives": count,\n               "tied_fraction": float(1 - len(np.unique(values)) / len(values))}\n        for k in (5, 10):\n            limit = min(k, len(order))\n            ideal = (1 / np.log2(np.arange(min(count, limit)) + 2)).sum()\n            row[f"ndcg@{k}"] = float((ranked[:limit] / np.log2(np.arange(limit) + 2)).sum() / ideal) if count else np.nan\n            row[f"recall@{k}"] = float(ranked[:limit].sum() / count) if count else np.nan\n        rows.append(row)\n        for rank, local in enumerate(order, 1):\n            if labels[local] == 1:\n                index = indices[local]\n                positives.append({"user": frame.iloc[index].user, "item": frame.iloc[index].item,\n                                  "query": query, "rank": rank, "hit5": int(rank <= 5),\n                                  "available": int(available[index])})\n    return pd.DataFrame(rows), pd.DataFrame(positives)\n\n\ndef summarize_strata(positive: pd.DataFrame, clean: pd.DataFrame) -> list[dict]:\n    """Functionality: compare masked-profile ranks with the clean reference for the same positive pairs.\n    Inputs: one scenario\'s positive table and the clean positive table.\n    Outputs: summaries grouped by available-feature count.\n    Logic: perform a one-to-one user/item join, then summarize users, positive pairs, rank changes, and\n    hit changes.\n    Constraints: do not remove candidate pairs; incomplete or duplicate joins raise an error."""\n    joined = positive.merge(clean[["user", "item", "rank", "hit5"]], on=["user", "item"],\n                            suffixes=("", "_clean"), validate="one_to_one")\n    output = []\n    for count, part in joined.groupby("available"):\n        output.append({"available": int(count), "users": int(part.user.nunique()), "positive_pairs": len(part),\n                       "mean_rank": float(part["rank"].mean()),\n                       "mean_rank_worsening": float((part["rank"] - part.rank_clean).mean()),\n                       "hit5": float(part.hit5.mean()), "hit5_change": float((part.hit5 - part.hit5_clean).mean())})\n    return output\n\n\ndef analyze(pairs: pd.DataFrame, predictions: dict, feature_arrays: dict, output: Path) -> dict:\n    """Functionality: analyze fixed predictions downloaded from Kaggle or produced by the experiment.\n    Inputs: test pairs, model/scenario predictions, scenario feature arrays, and output directory.\n    Outputs: metrics dictionary plus query evidence, bootstrap intervals, and candidate-gate reports.\n    Logic: compute primary metrics per scenario and use shared 2,000-draw bootstrap samples for repeat 0.\n    Constraints: no training or model selection occurs here; predictions must be finite; repeats 1–4 are\n    sensitivity analysis only; an engineering gate is not release approval."""\n    output.mkdir(parents=True, exist_ok=True)\n    metrics, queries, positive_tables, all_query_rows, all_positive_rows = {}, {}, {}, [], []\n    for scene, _ in feature_arrays.items():\n        available = np.isfinite(feature_arrays[scene]).sum(axis=1)\n        metrics[scene] = {}\n        for model in spec.PROTOCOL["models"]:\n            scores = predictions[scene + "__" + model]\n            if scores.shape != (len(pairs), 2) or not np.isfinite(scores).all():\n                raise ValueError(f"Invalid predictions: {scene}/{model}")\n            metrics[scene][model] = base.evaluate_scores(pairs, scores, "test")\n            for column, task in enumerate(("pref", "qual")):\n                observed = pairs["label_" + task].notna().to_numpy()\n                q, p = query_details(pairs.loc[observed], scores[observed, column], task, available[observed])\n                key = scene + "__" + model + "__" + task\n                queries[key], positive_tables[key] = q, p\n                all_query_rows.append(q.assign(scene=scene, model=model, task=task))\n                all_positive_rows.append(p.assign(scene=scene, model=model, task=task))\n                for metric in ("ndcg@5", "ndcg@10", "recall@5", "recall@10"):\n                    np.testing.assert_allclose(q[metric].mean(), metrics[scene][model][task][metric], rtol=0, atol=1e-12)\n                metrics[scene][model][task]["mean_tied_fraction"] = float(q.tied_fraction.mean())\n                metrics[scene][model][task]["queries_with_ties"] = int(q.tied_fraction.gt(0).sum())\n        LOGGER.info("METRICS %s completed", scene)\n    base.write_json(output / "metrics.json", metrics)\n    pd.concat(all_query_rows, ignore_index=True).to_csv(output / "query_metrics.csv.gz", index=False)\n    pd.concat(all_positive_rows, ignore_index=True).to_csv(output / "positive_ranks.csv.gz", index=False)\n    strata = {}\n    for scene in ("random_30_r0", "mixed_optional_r0", "optional_bundle_r0", "skills_only_r0"):\n        for model in spec.PROTOCOL["models"]:\n            for task in ("pref", "qual"):\n                key = scene + "__" + model + "__" + task\n                strata[key] = summarize_strata(positive_tables[key], positive_tables["clean_r0__" + model + "__" + task])\n    base.write_json(output / "completeness_strata.json", strata)\n    intervals, distributions, deltas = {}, {}, {}\n    r0_scenes = [scene for scene in feature_arrays if scene.endswith("_r0")]\n    for col, task in enumerate(("pref", "qual")):\n        observed = pairs["label_" + task].notna().to_numpy()\n        frame = pairs.loc[observed]\n        groups, ids = np.unique(frame["user" if task == "pref" else "item"], return_inverse=True)\n        counts = np.random.default_rng(42).multinomial(len(groups), np.full(len(groups), 1 / len(groups)), size=2000)\n        for scene in r0_scenes:\n            for model in spec.PROTOCOL["models"]:\n                prefix = scene + "__" + model + "__" + task\n                query = queries[prefix].set_index("query").loc[groups]\n                valid = query.positives.gt(0).to_numpy()\n                weights = counts[:, valid]\n                denominators = weights.sum(axis=1)\n                if (denominators == 0).any():\n                    raise ValueError("No positive ranking queries in bootstrap draw")\n                for metric in ("ndcg@5", "recall@5", "ndcg@10", "recall@10"):\n                    distributions[prefix + "__" + metric] = weights @ query.loc[valid, metric].to_numpy() / denominators\n                distributions[prefix + "__average_precision"] = previous.bootstrap_ap(\n                    frame["label_" + task].to_numpy(), predictions[scene + "__" + model][observed, col], ids, counts)\n            for metric in ("average_precision", "ndcg@5", "recall@5", "ndcg@10", "recall@10"):\n                for left, right in CONTRASTS:\n                    key = scene + "__" + left + "-" + right + "__" + task + "__" + metric\n                    diff = distributions[f"{scene}__{left}__{task}__{metric}"] - distributions[f"{scene}__{right}__{task}__{metric}"]\n                    deltas[key] = np.quantile(diff, [.025, .975]).tolist()\n            LOGGER.info("BOOTSTRAP task=%s scene=%s complete", task, scene)\n        for model in spec.PROTOCOL["models"]:\n            key = "primary_mean__" + model + "__" + task + "__ndcg@5"\n            distributions[key] = np.mean([distributions[f"{scene}_r0__{model}__{task}__ndcg@5"]\n                                           for scene in spec.PROTOCOL["primary_scenarios"]], axis=0)\n        for left, right in CONTRASTS:\n            key = "primary_mean__" + left + "-" + right + "__" + task + "__ndcg@5"\n            diff = distributions[f"primary_mean__{left}__{task}__ndcg@5"] - distributions[f"primary_mean__{right}__{task}__ndcg@5"]\n            deltas[key] = np.quantile(diff, [.025, .975]).tolist()\n    for key, values in distributions.items():\n        intervals[key] = np.quantile(values, [.025, .975]).tolist()\n    base.write_json(output / "bootstrap.json", {"seed": 42, "replicates": 2000, "intervals": intervals, "contrasts": deltas})\n    np.savez_compressed(output / "bootstrap_draws.npz", **distributions)\n    gates, sensitivity, clean_changes = {}, {}, {}\n    for model in spec.PROTOCOL["models"]:\n        gates[model] = {}\n        for task in ("pref", "qual"):\n            primary = float(np.mean([metrics[s + "_r0"][model][task]["ndcg@5"] for s in spec.PROTOCOL["primary_scenarios"]]))\n            reference = float(np.mean([metrics[s + "_r0"]["H"][task]["ndcg@5"] for s in spec.PROTOCOL["primary_scenarios"]]))\n            clean_ap = metrics["clean_r0"][model][task]["average_precision"] - metrics["clean_r0"]["H"][task]["average_precision"]\n            clean_ndcg = metrics["clean_r0"][model][task]["ndcg@5"] - metrics["clean_r0"]["H"][task]["ndcg@5"]\n            gates[model][task] = {"clean_ap_change": clean_ap, "clean_ndcg5_change": clean_ndcg,\n                                  "primary_ndcg5": primary, "primary_gain": primary - reference,\n                                  "point_estimate_gate_pass": bool(clean_ap >= -.02 and clean_ndcg >= -.02 and primary - reference >= .03)}\n            for scene in r0_scenes:\n                clean_changes[f"{scene}__{model}__{task}"] = {metric: metrics[scene][model][task][metric] - metrics["clean_r0"][model][task][metric]\n                    for metric in ("average_precision", "ndcg@5", "recall@5", "recall@10")}\n            for scenario in ("random_10", "random_30", "random_50", "mixed_optional", "job_random_30"):\n                sensitivity[f"{scenario}__{model}__{task}"] = {metric: [min(metrics[f"{scenario}_r{r}"][model][task][metric] for r in range(5)),\n                    max(metrics[f"{scenario}_r{r}"][model][task][metric] for r in range(5))]\n                    for metric in ("average_precision", "ndcg@5", "recall@5")}\n    base.write_json(output / "gates.json", gates)\n    base.write_json(output / "mask_sensitivity.json", sensitivity)\n    base.write_json(output / "change_from_clean.json", clean_changes)\n    return metrics\n'
module = types.ModuleType('jobrec_missing_eval')
exec(compile(source, 'jobrec_missing_eval.py', 'exec'), module.__dict__)
sys.modules['jobrec_missing_eval'] = module


In [ ]:
"""Responsibilities: execute the approved v4 missing-profile training study and verify its historical baseline.
Implementation: verify fixed parent-artifact hashes, rebuild masked-profile features, fit eight rankers,
and save predictions for a unified analysis.
Related Modules: jobrec_v1/jobrec_v2 provide frozen historical helpers; jobrec_missing defines missingness;
jobrec_missing_eval analyzes outputs.

Declaration Index:
- verify_history: verify the identities of v3 rankers and recorded evaluation artifacts.
- prepare_scenes: rebuild registered missing-profile scenes from hidden source fields.
- train_model: fit one registered pair of preference/qualification rankers.
- main: execute the staged data, training, and evaluation workflow.

Variable Index:
- LOGGER: stage-level logging without credentials or profile text; experiment settings come from
  spec.PROTOCOL and spec.RANKER_CONFIG.

Constraints: preserve historical models, splits, labels, and tree parameters; each failed stage is
recorded and re-raised, with no retry or fallback."""

import hashlib
import importlib.metadata
import json
import logging
import shutil
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

import jobrec_v1 as base
import jobrec_v2 as previous
import jobrec_missing as spec
import jobrec_missing_eval as evaluation

LOGGER = logging.getLogger("jobrec_v4")


def verify_history(folder: Path) -> None:
    """Functionality: verify the identity of historical v3 rankers and result files.
    Inputs: parent artifact directory. Outputs: none.
    Logic: compare each file's SHA256 with the frozen protocol manifest.
    Constraints: any mismatch raises an error; a newer file with the same name is not accepted."""
    for name, expected in spec.HISTORICAL.items():
        if hashlib.sha256((folder / name).read_bytes()).hexdigest() != expected:
            raise ValueError(f"v3 historical artifact changed: {name}")
    LOGGER.info("Historical v3 artifacts verified")


def prepare_scenes(pairs: pd.DataFrame, users: pd.DataFrame, jobs: pd.DataFrame,
                   clean: np.ndarray, output: Path) -> dict:
    """Functionality: construct the fixed missing-profile scenarios.
    Inputs: pair/user/job tables, the complete reference feature matrix, and the output directory.
    Outputs: a mapping from scenario keys to feature matrices for all pairs.
    Logic: mask entity fields before recomputing features, then save entity IDs, masks, and audit summaries.
    Constraints: the clean reconstruction must match element by element; original profiles are not written
    as scenario output; training masks use a separate namespace; missing feature values remain NaN and do
    not fill missing labels."""
    data, masks, audits = {}, {}, {}
    settings = spec.scenarios() + [("training_mix", 0)]
    for scenario, repeat in settings:
        key = f"{scenario}_r{repeat}"
        um, jm, assignments = spec.make_masks(users, jobs, scenario, repeat,
                                              "training" if scenario == "training_mix" else "evaluation")
        hidden_u = spec.hide_profiles(users, um, spec.USER_BLOCKS)
        hidden_j = spec.hide_profiles(jobs, jm, spec.JOB_BLOCKS)
        matrix = spec.features(pairs, hidden_u, hidden_j)
        if scenario == "clean":
            np.testing.assert_array_equal(matrix, clean)
        data[key] = matrix
        masks[key + "__users"], masks[key + "__jobs"] = um, jm
        audits[key] = {"user_blocks_hidden": um.sum(axis=0).tolist(), "job_blocks_hidden": jm.sum(axis=0).tolist(),
                       "feature_missing_counts": np.isnan(matrix).sum(axis=0).tolist(),
                       "user_assignments": pd.Series(assignments).value_counts().to_dict()}
        if scenario == "training_mix":
            train_users = np.unique(pairs.loc[pairs.split.eq("train"), "u_idx"])
            audits[key]["labeled_training_user_assignments"] = pd.Series(np.asarray(assignments)[train_users]).value_counts().to_dict()
        LOGGER.info("FEATURES scene=%s missing=%s", key, audits[key]["feature_missing_counts"])
    np.savez_compressed(output / "masks.npz", **masks)
    users[["student_id"]].to_csv(output / "candidate_ids.csv", index=False)
    jobs[["job_id"]].to_csv(output / "job_ids.csv", index=False)
    base.write_json(output / "mask_audit.json", audits)
    np.savez_compressed(output / "training_features.npz", clean=data["clean_r0"], mixed=data["training_mix_r0"])
    return data


def train_model(model_name: str, pairs: pd.DataFrame, data: dict, folder: Path) -> list:
    """Functionality: fit one preregistered bidirectional comparison variant.
    Inputs: variant name, pair table, scenario feature matrices, and model output directory.
    Outputs: the two selected, reloaded LightGBM Boosters.
    Logic: A uses complete profiles; B/C/D share the fixed mixed-profile training input. Group rows by
    query for each task and select from the 20-to-200 tree grid using the equal mean AP over three
    validation scenarios.
    Constraints: fit on train only and select on validation only; do not duplicate pairs; resolve ties in
    favor of the earlier iteration; save and reload the chosen model."""
    import lightgbm as lgb

    outputs, history = [], {}
    train_features = spec.feature_view(data["clean_r0" if model_name == "A" else "training_mix_r0"], model_name)
    for task in ("pref", "qual"):
        label, group = "label_" + task, "user" if task == "pref" else "item"
        train = pairs.loc[pairs.split.eq("train") & pairs[label].notna()].sort_values(group, kind="stable")
        validation = pairs.loc[pairs.split.eq("validation") & pairs[label].notna()]
        sizes = train.groupby(group, sort=False).size().to_numpy()
        if sizes.sum() != len(train):
            raise ValueError("Invalid query grouping")
        model = lgb.LGBMRanker(**spec.RANKER_CONFIG)
        model.fit(train_features[train.index], train[label].astype(int), group=sizes)
        records = []
        for iteration in spec.PROTOCOL["tree_grid"]:
            aps = {}
            for scenario in spec.PROTOCOL["validation_scenarios"]:
                matrix = spec.feature_view(data[scenario + "_r0"][validation.index], model_name)
                scores = model.booster_.predict(matrix, num_iteration=iteration, num_threads=2)
                aps[scenario] = float(average_precision_score(validation[label], scores))
            records.append({"iteration": iteration, "scenario_ap": aps, "mean_ap": float(np.mean(list(aps.values())))})
        selected = max(records, key=lambda row: row["mean_ap"])
        target = folder / (task + ".txt")
        model.booster_.save_model(str(target), num_iteration=selected["iteration"])
        booster = lgb.Booster(model_file=str(target))
        outputs.append(booster)
        history[task] = {"selection": selected, "history": records, "training_pairs": len(train),
                         "training_queries": len(sizes), "actual_params": model.get_params(),
                         "features": train_features.shape[1]}
        LOGGER.info("TRAIN model=%s task=%s selected=%s", model_name, task, selected)
    base.write_json(folder / "history.json", history)
    return outputs


def main() -> None:
    """Functionality: run the complete v4 experiment.
    Inputs: mounted Kaggle parent kernels, network access, and the approved fixed protocol.
    Outputs: the `jobrec_v4` artifacts under `/kaggle/working`.
    Logic: verify inputs, initialize the environment, reconstruct data and the clean baseline, diagnose
    missingness, fit eight rankers, and evaluate; log each stage and persist failure status.
    Constraints: retain the historical environment setup. Trees train on CPU with two threads; do not
    claim GPU training. This study does not train text or interview models."""
    output = Path("/kaggle/working/jobrec_v4")
    output.mkdir(parents=True, exist_ok=True)
    logging.basicConfig(level=logging.INFO, force=True, format="%(asctime)s %(levelname)s %(message)s",
                        handlers=[logging.StreamHandler(sys.stdout), logging.FileHandler(output / "execution.log")])
    started, stage = time.monotonic(), "protocol"
    base.write_json(output / "protocol.json", spec.PROTOCOL)
    base.write_json(output / "protocol_hash.json", {"sha256": hashlib.sha256((output / "protocol.json").read_bytes()).hexdigest()})
    try:
        roots = list(Path("/kaggle/input").rglob("jobrec_v1/config.json"))
        histories = list(Path("/kaggle/input").rglob("jobrec_v3/lambdarank/pref.txt"))
        if len(roots) != 1 or len(histories) != 1:
            raise ValueError(f"Expected one v1/v3 parent; found {len(roots)}/{len(histories)}")
        parent, history_parent = roots[0].parent, histories[0].parent.parent
        previous.verify_inputs(parent)
        verify_history(history_parent)
        stage = "environment"
        base.bootstrap(output)
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "lightgbm==4.6.0"], check=True)
        import lightgbm as lgb

        base.write_json(output / "ranker_environment.json", {"lightgbm": importlib.metadata.version("lightgbm"),
                        "compute": "CPU, n_jobs=2; no embedding or neural training"})
        stage = "data"
        frames = base.download_data(output)
        if json.loads((output / "data_manifest.json").read_text()) != json.loads((parent / "data_manifest.json").read_text()):
            raise ValueError("Source data changed")
        pairs, users, jobs, _, clean = base.prepare_data(frames, output)
        pd.testing.assert_frame_equal(pairs, pd.read_csv(parent / "pair_splits.csv"), check_dtype=False)
        np.testing.assert_array_equal(spec.features(pairs, users, jobs), clean)
        test = pairs.loc[pairs.split.eq("test")].copy()
        test_ids = test.index.to_numpy()
        test.to_csv(output / "test_pairs.csv", index=False)
        models, predictions = {}, {}
        historical_folder = output / "H"
        historical_folder.mkdir()
        models["H"] = []
        for task in ("pref", "qual"):
            shutil.copy2(history_parent / "lambdarank" / (task + ".txt"), historical_folder / (task + ".txt"))
            models["H"].append(lgb.Booster(model_file=str(historical_folder / (task + ".txt"))))
        clean_scores = np.column_stack([model.predict(clean[test_ids], num_threads=2) for model in models["H"]])
        old = pd.read_csv(history_parent / "test_predictions.csv")
        pd.testing.assert_frame_equal(test[["user", "item"]].reset_index(drop=True), old[["user", "item"]])
        np.testing.assert_allclose(clean_scores, old[["lambdarank_taskwise_pref", "lambdarank_taskwise_qual"]].to_numpy(), rtol=1e-12, atol=1e-12)
        reference_metrics = json.loads((history_parent / "metrics.json").read_text())["lambdarank_taskwise"]
        actual = base.evaluate_scores(test, clean_scores, "test")
        for task in actual:
            for metric, value in actual[task].items():
                np.testing.assert_allclose(value, reference_metrics[task][metric], rtol=0, atol=1e-12)
        base.write_json(output / "baseline_verification.json", {"features_exact": True, "predictions_tolerance": 1e-12,
                        "metrics_tolerance": 1e-12, "metrics": actual, "history_hashes": spec.HISTORICAL})
        LOGGER.info("BASELINE clean features, predictions and metrics verified")
        stage = "mask_features"
        data = prepare_scenes(pairs, users, jobs, clean, output)
        scenes = {key: matrix[test_ids] for key, matrix in data.items() if key != "training_mix_r0"}
        np.savez_compressed(output / "test_features.npz", **scenes)
        stage = "historical_diagnostics"
        diagnostics = {}
        for scene, matrix in scenes.items():
            values = np.column_stack([model.predict(matrix, num_threads=2) for model in models["H"]])
            predictions[scene + "__H"] = values
            diagnostics[scene] = base.evaluate_scores(test, values, "test")
        base.write_json(output / "historical_diagnostics.json", diagnostics)
        LOGGER.info("HISTORICAL missing diagnostics complete; training protocol unchanged")
        for name in "ABCD":
            stage = "train_" + name
            folder = output / name
            folder.mkdir()
            models[name] = train_model(name, pairs, data, folder)
            for scene, matrix in scenes.items():
                inputs = spec.feature_view(matrix, name)
                predictions[scene + "__" + name] = np.column_stack([model.predict(inputs, num_threads=2) for model in models[name]])
        np.savez_compressed(output / "predictions.npz", **predictions)
        stage = "evaluation"
        evaluation.analyze(test.reset_index(drop=True), predictions, scenes, output)
        base.write_json(output / "status.json", {"status": "complete", "elapsed_seconds": time.monotonic() - started,
                        "trained_models": 8, "scenarios_including_mask_repeats": len(scenes)})
        LOGGER.info("JOBREC_V4_COMPLETE")
    except Exception as error:
        base.write_json(output / "status.json", {"status": "failed", "stage": stage, "error": str(error)})
        LOGGER.exception("v4 failed at %s; no fallback/retry", stage)
        raise


if __name__ == "__main__":
    main()
